## Cleansing approach for Diffusion-DB using vector search

The data cleansing of DiffusionDB-2M has been disclosed in public notebooks and discussions.

https://www.kaggle.com/code/shoheiazuma/diffusiondb-data-cleansing/notebook
https://www.kaggle.com/competitions/stable-diffusion-image-to-prompts/discussion/398529

I achieved this through very simple rule-based filtering and filtering based on the similarity of prompt vectors. For the evaluation of vector similarity, I used the faiss vector search library, which is used for recommendation and similar image search.

I believe that this search technique will be important in this competition.

In [ ]:
%pip install faiss-gpu

In [ ]:
import sys
import re
import faiss
import torch
import numpy as np
import polars as pl
from pathlib import Path
import torch.nn.functional as F
from tqdm.notebook import tqdm
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append("/kaggle/input/sentence-transformers-222/sentence-transformers")
from sentence_transformers import SentenceTransformer

## Rule-Based Filtering

In [ ]:
def check_string(string: str) -> bool:
    # Checks if the given string contains any character other than alphanumeric characters, comma, dot, hyphen or whitespace
    return bool(re.search(r'[^A-Za-z0-9,.\\-\\s]', string))

In [ ]:
# Load data from a Parquet file
# For the purpose of illustration, the amount of data will be reduced
pldf = pl.read_parquet("/kaggle/input/diffusiondb-metadata/metadata.parquet", columns=['image_name', 'prompt', 'width', 'height'])

# Select only those images whose width and height fall between 256 and 768 pixels
pldf = pldf.filter(pl.col("width").is_between(256, 768) & pl.col("height").is_between(256, 768))

# Select only those prompts that have five or more words 
pldf = pldf.filter(pl.col("prompt").str.split(" ").apply(lambda x: len(x)>=5))

# Select only those prompts that are not blank, NULL, null, or NaN
pldf = pldf.filter(~pl.col("prompt").str.contains('^(?:\s*|NULL|null|NaN)$'))


pldf = pldf.filter(pl.col("prompt").apply(check_string))
pldf.glimpse()

In [ ]:
#For the purpose of illustration, we will reduce the amount of data
pldf = pldf[:10000]

## Vectorize using SentenceTransformers

In [ ]:
model = SentenceTransformer("/kaggle/input/sentence-transformers-222/all-MiniLM-L6-v2")
vector = model.encode(pldf["prompt"].to_numpy(), batch_size=512, show_progress_bar=True, device="cuda", convert_to_tensor=True)

## Similarity filtering using vector search

In [ ]:
threshold = 0.80  # Set the threshold for similarity.
n_neighbors = 1000  # Set the number of neighbors to consider.

# Perform batch processing because processing all data at once may cause resource shortage.
batch_size = 1000  # Set the batch size (i.e., the number of data items to be processed at once).
similar_vectors = []  # Create an empty list to store similar vectors.

In [ ]:
# Create an IndexFlatIP index using the Faiss library
# The term 'IP' represents the Inner Product, 
# which is equivalent to cosine similarity as it involves taking the dot product of normalized vectors.
resources = faiss.StandardGpuResources()
index = faiss.IndexIVFFlat(faiss.IndexFlatIP(384), 384, 5, faiss.METRIC_INNER_PRODUCT)
gpu_index = faiss.index_cpu_to_gpu(resources, 0, index)

# Normalize the input vector and add it to the IndexFlatIP 
gpu_index.train(F.normalize(vector).cpu().numpy())
gpu_index.add(F.normalize(vector).cpu().numpy())

In [ ]:
for i in tqdm(range(0, len(vector), batch_size)):
    # Get the target batch for processing.
    batch_data = vector.cpu().numpy()[i:i + batch_size]
    # Neighborhood search based on cosine similarity.
    similarities, indices = gpu_index.search(batch_data, n_neighbors)
    
    # Extract indexes and similarities of data to be deleted.
    for j in range(similarities.shape[0]):
        close_vectors = indices[j, similarities[j] >= threshold] 
        index_base = i
        # Get only the similar vectors that exclude itself
        close_vectors = close_vectors[close_vectors != index_base + j]  
        similar_vectors.append((index_base + j, close_vectors))


### Drop Similarity Data

In [ ]:
pldf = pldf.with_columns(pl.Series(values=list(range(len(pldf))), name="index"))
pldf = pldf.filter(~pl.col("index").is_in(np.unique(np.concatenate([x for _, x in similar_vectors])).tolist()))

In [ ]:
for i, _ in tqdm(enumerate(range(1, 2000, 100)), total=20):
    image_dir = Path("/kaggle/input/diffusiondb-2m-part-{:04d}-to-{:04d}-of-2000/".format(i * 100 + 1, (i + 1) * 100))
    pldf = pldf.with_columns(
        pl.when(pl.col("image_name").is_in([str(file_path.name) for file_path in image_dir.glob("*.png")]))
        .then(str(image_dir) + "/" + pl.col("image_name"))
        .otherwise(pl.col("image_name"))
        .alias("image_name")
    )

In [ ]:
pldf.select(pl.col("image_name", "prompt")).write_csv("diffusiondb.csv")
pldf.select(pl.col("image_name", "prompt")).head()